# Lab L1.2 · Credit-risk feature engineering, leakage audit and a reusable preprocessing pipeline
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 1 · ⏱ 90 min · **Level 2 · Guided starter (fill in the blanks) · ★ recommended level**

Engineer bank-grade credit features, prove which columns leak, split out-of-time, and ship a fitted scikit-learn pipeline that Day 3 and Day 4 reuse.

### How this notebook works
- Every `✏️ BLANK n` marks code **you** write. Inline blanks look like `____`; block blanks say `# ...your code here...`.
- The markdown cell above each code cell explains **what** to write, **where**, and **why**.
- Stuck > 5 min? Open the 💡 hint in the HTML lab guide, then ask your trainer. The Level 3 solution is released after the lab.

### Scenario
**Horizon Gulf Bank (HGB)** is a fictional UAE retail bank with 250,000 customers. Its Board approved **Project MIZAN** — an AI decisioning platform covering credit, fraud, collateral valuation, cash logistics, customer retention and document automation. You are a junior ML practitioner in the MIZAN squad.

The CRO wants an honest **probability-of-default (PD)** model for retail lending. Your job today is the part that decides whether the model can be trusted: which columns are legitimate, which leak, which are *sensitive* (fairness), how the data is split in time, and a preprocessing pipeline that behaves identically in training and in production.

**Progressive architecture:** L3.1 (classification), L4.1 (SVR) and L4.2 (MLP) all load your feature table and pipeline.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load and assign a role to every column

**Why this matters.** Before engineering anything, decide what each column *is*. This **column-role contract** prevents the two most common credit-model failures: leakage and using protected attributes. Gender and nationality are kept for **fairness monitoring** but never used as inputs.

#### ✏️ What to write in the next cell
- **Blank 1** *(inline)* — columns that only exist AFTER the 12-month outcome

In [ ]:
loans = pd.read_csv(DATA / "loan_applications.csv", parse_dates=["application_date"])
ID_COLS = ["application_id", "customer_id"]
TARGET = "default_12m"
POST_OUTCOME = ____  # ✏️ B1
SENSITIVE = ["gender", "nationality_group"]          # fairness monitoring only — never model inputs
CANDIDATES = [c for c in loans.columns if c not in ID_COLS + [TARGET] + POST_OUTCOME + SENSITIVE + ["application_date", "risk_grade"]]
print(len(loans), "loans | default rate:", round(loans[TARGET].mean(), 4))
print(loans.groupby(loans.application_date.dt.year)[TARGET].mean().round(4))
print("candidates:", CANDIDATES)

**✔ Checkpoint.** 300,000 loans, default rate ≈ 7.5%, 2020 ≈ 12% (COVID). ~19 candidate columns.

## Step 2 · Quantitative leakage screen

**Why this matters.** Opinions about leakage are not evidence. A **single-feature AUC** screen ranks every numeric column by how well it alone separates defaulters. Anything near 1.0 is either a miracle or a leak — and in banking it is always a leak.

#### ✏️ What to write in the next cell
- **Blank 2** *(inline)* — AUC of the raw column used as a score

In [ ]:
from sklearn.metrics import roc_auc_score
s = loans.sample(50_000, random_state=RNG)
rows = []
for col in s[CANDIDATES + POST_OUTCOME].select_dtypes("number").columns:
    x = s[col].fillna(s[col].median())
    auc = ____  # ✏️ B2
    rows.append((col, max(auc, 1 - auc), col in POST_OUTCOME))
screen = pd.DataFrame(rows, columns=["feature", "single_feature_auc", "post_outcome"]).sort_values("single_feature_auc", ascending=False)
screen["suspect"] = screen.single_feature_auc > 0.90
assert set(screen.loc[screen.suspect, "feature"]) <= set(POST_OUTCOME), "unexpected leak among candidates!"
screen.round(3)

**✔ Checkpoint.** `max_days_past_due`, `collection_calls` and `loss_amount_aed` top the list with AUC ≈ 0.95–1.0; the best legitimate feature (`aecb_score`) ≈ 0.75, then prior delinquencies ≈ 0.71.

## Step 3 · Engineer credit-domain features

**Why this matters.** Raw columns rarely express risk the way credit officers think about it. **Loan-to-income**, **disposable income after obligations**, **bureau-file status** and **inquiry intensity** are standard underwriting concepts. Writing them as a *pure function* (no fitting) means it is safe to apply to any split and to production data.

#### ✏️ What to write in the next cell
- **Blank 3** *(inline)* — exposure relative to yearly income
- **Blank 4** *(code block, ≈1 lines)* — disposable income = salary minus existing obligations minus the new instalment
- **Blank 5** *(inline)* — band edges Poor/Fair/Good/Very Good/Excellent

In [ ]:
def add_features(d: pd.DataFrame) -> pd.DataFrame:
    """Pure (stateless) credit features — identical in training and production."""
    d = d.copy()
    annual_salary = d.monthly_salary_aed * 12
    d["loan_to_income"] = ____  # ✏️ B3
    d["installment_to_salary"] = d.monthly_installment_aed / d.monthly_salary_aed
    # ✏️ BLANK 4 — write ≈1 line(s): disposable income = salary minus existing obligations minus the new instalment
    # ...your code here...
    d["log_salary"] = np.log1p(d.monthly_salary_aed)
    d["log_loan_amount"] = np.log1p(d.loan_amount_aed)
    d["aecb_missing"] = d.aecb_score.isna().astype(int)
    d["aecb_band"] = pd.cut(d.aecb_score, bins=____,  # ✏️ B5
                            labels=["Poor", "Fair", "Good", "Very_Good", "Excellent"]).cat.add_categories("No_File").fillna("No_File").astype(str)
    d["thin_file_young"] = ((d.age < 28) & d.aecb_score.isna()).astype(int)
    d["employment_years"] = d.months_at_employer / 12
    d["inquiries_x_dq"] = d.num_bureau_inquiries_6m * (1 + d.num_prior_delinquencies_24m)
    d["app_month"] = d.application_date.dt.month
    return d

feat = add_features(loans)
print(feat.groupby("aecb_band")[TARGET].agg(["mean", "size"]).sort_values("mean").round(4))

**✔ Checkpoint.** Default rate rises steadily from *Excellent* (< 1%) to *Poor* (≈ 30%); *No_File* sits well above average — validating the MNAR flag from Lab 1.1.

## Step 4 · Out-of-time train / validation / test split

**Why this matters.** A PD model is used on **future** applicants. Regulators and model validators expect an **out-of-time (OOT)** test. We train on 2019 → mid-2023, tune on the following year, and keep mid-2024 → 2025 untouched until the very end.

#### ✏️ What to write in the next cell
- **Blank 6** *(inline)* — oldest loans
- **Blank 7** *(inline)* — the following 12 months

In [ ]:
feat["split"] = np.select(
    [feat.application_date < "2023-07-01", feat.application_date < "2024-07-01"],
    [____, ____], default="test")  # ✏️ B6; B7
print(feat.groupby("split")[TARGET].agg(rows="size", default_rate="mean").loc[["train", "valid", "test"]].round(4))
assert feat.groupby("split").application_date.max()["train"] < feat.groupby("split").application_date.min()["valid"]

**✔ Checkpoint.** ≈ 208k / 46k / 46k rows. Train default rate (≈7.9%) is higher than valid/test (≈6.7%) because it contains 2020 — real-world drift.

## Step 5 · Build the preprocessing ColumnTransformer

**Why this matters.** Different columns need different treatment. A `ColumnTransformer` bundles all of it so that **`fit` only ever sees training rows** and `transform` is applied identically to validation, test and production. This single object eliminates train-test contamination.

#### ✏️ What to write in the next cell
- **Blank 8** *(inline)* — median is robust to skew; indicator keeps 'was missing'
- **Blank 9** *(inline)* — unknown-safe; groups categories < 1% together
- **Blank 10** *(code block, ≈1 lines)* — fit the preprocessor on TRAIN rows only and transform them into Xt

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer

NUM_LOG = ["loan_amount_aed", "monthly_salary_aed", "existing_monthly_obligations_aed", "monthly_installment_aed"]
NUM = ["tenure_months", "interest_rate", "age", "months_at_employer", "num_dependents", "aecb_score", "num_bureau_inquiries_6m",
       "num_prior_delinquencies_24m", "debt_burden_ratio", "loan_to_income", "installment_to_salary", "disposable_income_aed", "inquiries_x_dq"]
CAT = ["product", "emirate", "employment_type", "residence_type", "aecb_band"]
ORD = ["employer_category"]
BIN = ["salary_transfer", "aecb_missing", "thin_file_young"]

log_pipe = Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("scale", StandardScaler())])
num_pipe = Pipeline([("impute", ____), ("scale", StandardScaler())])  # ✏️ B8
cat_pipe = ____  # ✏️ B9
ord_pipe = OrdinalEncoder(categories=[["Government", "A", "B", "C", "Unlisted"]], handle_unknown="use_encoded_value", unknown_value=5)

preprocessor = ColumnTransformer([
    ("log", log_pipe, NUM_LOG), ("num", num_pipe, NUM), ("cat", cat_pipe, CAT), ("ord", ord_pipe, ORD), ("bin", "passthrough", BIN)],
    verbose_feature_names_out=False)

train = feat[feat.split == "train"]
# ✏️ BLANK 10 — write ≈1 line(s): fit the preprocessor on TRAIN rows only and transform them into Xt
# ...your code here...
names = preprocessor.get_feature_names_out()
print(Xt.shape); print(list(names[:15]))

**✔ Checkpoint.** Roughly `(207700, 46)` — scaled numerics, one-hot columns and a missing-indicator for `aecb_score`.

## Step 6 · Prove the pipeline is honest: baseline model with and without leakage

**Why this matters.** A feature pipeline is only finished when a model trained on it gives a **credible** result. We compare the honest feature set with a version that sneaks the post-outcome columns back in — the gap is the price of leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
valid = feat[feat.split == "valid"]
honest = Pipeline([("prep", clone(preprocessor)), ("clf", LogisticRegression(max_iter=2000))]).fit(train, train[TARGET])
auc_honest = roc_auc_score(valid[TARGET], honest.predict_proba(valid)[:, 1])

leaky_prep = ColumnTransformer(preprocessor.transformers + [("leak", StandardScaler(), ["max_days_past_due", "collection_calls"])],
                               verbose_feature_names_out=False)
leaky = Pipeline([("prep", leaky_prep), ("clf", LogisticRegression(max_iter=2000))]).fit(train, train[TARGET])
auc_leaky = roc_auc_score(valid[TARGET], leaky.predict_proba(valid)[:, 1])
print(f"validation AUC — honest: {auc_honest:.3f} | with post-outcome leakage: {auc_leaky:.3f}")

**✔ Checkpoint.** Honest ≈ 0.78–0.82; leaky ≈ 0.99+. Write down your honest baseline — Lab 3.1 must beat it.

## Step 7 · Feature-selection fundamentals

**Why this matters.** More features ≠ better model. **Filter methods** are fast first passes: drop near-constant columns, flag highly correlated pairs (redundant, unstable coefficients), and rank by **mutual information** (captures non-linear dependence).

#### ✏️ What to write in the next cell
- **Blank 11** *(inline)* — redundancy threshold
- **Blank 12** *(inline)* — MI between each feature and the target

In [ ]:
from sklearn.feature_selection import mutual_info_classif
Xdf = pd.DataFrame(Xt, columns=names)
low_var = Xdf.columns[Xdf.var() < 0.01].tolist()
print("near-constant:", low_var)
corr = Xdf.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()
print("highly correlated pairs:\n", pairs[pairs > ____].sort_values(ascending=False).round(3))  # ✏️ B11
idx = np.random.default_rng(RNG).choice(len(Xdf), 40_000, replace=False)
mi = pd.Series(____, index=names)  # ✏️ B12
mi.sort_values(ascending=False).head(12).round(4)

**✔ Checkpoint.** The three ‘no bureau file’ columns (`aecb_missing`, `aecb_band_No_File`, the imputer indicator) correlate at 1.0 — the redundancy predicted in step 5. `interest_rate` ↔ `product_Credit_Card` ≈ 0.99. Bureau score, interest rate and delinquencies top the MI ranking.

## Step 8 · Persist the data contract for Day 3 & Day 4

**Why this matters.** Downstream labs must not re-derive features differently. We save the **feature table**, the **fitted pipeline** and a **JSON spec** of column roles — three artefacts that together form the contract between data preparation and modelling.

#### ✏️ What to write in the next cell
- **Blank 13** *(inline)* — round-trip test of the saved artefact

In [ ]:
import joblib, json
feat.to_parquet(CKPT / "loan_features.parquet", index=False)
joblib.dump(preprocessor, CKPT / "loan_preprocessor.joblib")
spec = {"target": TARGET, "num_log": NUM_LOG, "num": NUM, "cat": CAT, "ord": ORD, "bin": BIN,
        "sensitive": SENSITIVE, "post_outcome": POST_OUTCOME, "honest_valid_auc": round(auc_honest, 4),
        "split_rule": "train < 2023-07-01 <= valid < 2024-07-01 <= test"}
(CKPT / "loan_feature_spec.json").write_text(json.dumps(spec, indent=2))
reloaded = ____  # ✏️ B13
assert reloaded.transform(valid).shape[1] == Xt.shape[1]
print("saved:", [p.name for p in CKPT.glob("loan_*")])

**✔ Checkpoint.** Three `loan_*` files in `checkpoints/`; the reload assert passes.

## ✅ Lab complete
The credit data contract is published. **Tomorrow** you’ll learn the training discipline that turns features like these into well-generalising models.